# CS231n Lecture 5: Convolutional Neural Networks (CNNs) — Study Notes

---

## Table of Contents
* **[Section 0: Warm-up & Historical Context](#section-0-warm-up--historical-context)**
* **[Section 1: The Space Trap — Fully Connected vs. Convolutional Layers](#section-1-the-space-trap--fully-connected-vs-convolutional-layers)**
* **[Section 2: The 2D Convolutional Operator & Spatial Mathematics](#section-2-the-2d-convolutional-operator--spatial-mathematics)**
* **[Section 3: Receptive Fields & Downsampling Mechanics](#section-3-receptive-fields--downsampling-mechanics)**
* **[Section 4: Pooling Layers — Max vs. Average](#section-4-pooling-layers--max-vs-average)**
* **[Section 5: Mathematical Foundations — Translation Equivariance](#section-5-mathematical-foundations--translation-equivariance)**
* **[Section 6: Implementation from Scratch & PyTorch API](#section-6-implementation-from-scratch--pytorch-api)**
* **[Section 7: Summary, Key Takeaways, & Pitfalls](#section-7-summary-key-takeaways--pitfalls)**

---

## Section 0: Warm-up & Historical Context

To understand the sudden shift in computer vision in the early 2010s, one must appreciate the history of feature extraction. Historically, raw pixel values are highly sensitive to small variations (illumination, camera angle, minor shifts). Thus, computer vision engineers designed hand-crafted feature extractors by hand to capture robust descriptors of images.

### Pre-2012: Hand-Crafted Feature Paradigms
1. **Color Histograms:** Counts the distribution of pixel colors, completely throwing away spatial layouts.
2. **Histogram of Oriented Gradients (HOG):** Measures the distribution of local edge orientations. Highly effective for shape and pedestrian detection.
3. **Bag of Words (BoW):** Inspired by NLP, this clusters local scale-invariant feature transform (SIFT) descriptors into a "visual vocabulary" and builds frequency histograms.

```
+-------------+      +-------------------+      +-------------------+      +------------------+
|  Raw Image  | ---> |   Hand-Designed   | ---> | Feature Histogram | ---> | Linear Classifier|
|             |      | Feature Extractor |      | (e.g. HOG, SIFT)  |      |   (e.g. SVM)     |
+-------------+      +-------------------+      +-------------------+      +------------------+
```

### The End-to-End Deep Learning Paradigm Shift
In **2012**, ImageNet (specifically the success of **AlexNet**) changed the computer vision paradigm. Instead of humans designing feature extractors by hand and feeding the output features into a linear classifier (like an SVM), neural networks merged feature extraction and classification into a **single, unified, end-to-end learnable system**. 

The network learns both the hierarchical feature maps (edges, textures, shapes, high-level parts) and the final linear classifier simultaneously through backpropagation and gradient descent:

```
+-------------+      +-------------------------------------------------+      +------------------+
|  Raw Image  | ---> |  Hierarchical Learnable Convolutional Layers    | ---> | Softmax / Linear |
|             |      | (Learns filters directly via Gradient Descent)  |      |    Classifier    |
+-------------+      +-------------------------------------------------+      +------------------+
```

### Modern Context (Post-2021 Era)
While **Vision Transformers (ViTs)** have recently emerged as state-of-the-art general-purpose scaling engines for computer vision (displacing pure CNN architectures on very large datasets due to lack of local inductive biases), **Convolutional Neural Networks remain highly relevant**. CNNs possess strong inductive spatial biases (local translation equivariance and local spatial locality) that allow them to train quickly and efficiently on small-to-medium-sized datasets without requiring massive pretraining, making them foundational to modern computer vision systems.

---

## Section 1: The Space Trap — Fully Connected vs. Convolutional Layers

### 1. Intuition: Preserving the 2D Spatial Structure
An image is naturally a 3D tensor: $X \in \mathbb{R}^{H \times W \times C}$, where $H$ is the height, $W$ is the width, and $C$ is the number of color channels (e.g., $C=3$ for RGB, $C=1$ for Grayscale).

Standard Multi-Layer Perceptrons (MLPs) or **Fully Connected (FC)** layers require inputs to be flattened into a single-dimensional vector $x \in \mathbb{R}^{D}$, where $D = H \times W \times C$. 

This flattening process is highly problematic:
* **Loss of Spatial Proximity:** Flattening completely destroys the 2D grid structure. A pixel at spatial location $(i, j)$ is adjacent to $(i, j+1)$ and $(i+1, j)$. Once flattened, these pixels are mapped to arbitrary, distant indices in the vector, making it highly difficult for the network to capture local geometric shapes or textures.
* **Parameter Explosion:** A fully connected neuron connects to *every* single input element. The number of parameters scales linearly with the input dimension, leading to severe memory constraints.

### 2. Parameter Growth Analysis: FC vs. Conv
Let's analyze the mathematical scaling of parameters. 

Suppose we have an input image of height $H$, width $W$, and $C_{in}$ channels. We connect it to a hidden layer containing $M$ units (neurons).

#### Fully Connected Layer Scaling
The number of weights in a single FC layer is given by:
$$\text{Params}_{\text{FC}} = (H \cdot W \cdot C_{in}) \times M + M \quad (\text{Weights + Biases})$$

* **Modest Input Example ($32 \times 32 \times 3$ image, CIFAR-10, $M=1000$):**
  $$\text{Params}_{\text{FC}} = (32 \times 32 \times 3) \times 1000 + 1000 = 3,072,000 + 1000 \approx 3.07 \text{ million parameters}$$
* **HD Input Example ($1000 \times 1000 \times 3$ image, $M=1000$):**
  $$\text{Params}_{\text{FC}} = (1000 \times 1000 \times 3) \times 1000 + 1000 = 3,000,000 \times 1000 + 1000 \approx \mathbf{3.00 \text{ billion parameters!}}$$

This parameter explosion is completely unscalable for high-resolution images, leading to rapid overfitting and hardware memory exhaustion.

#### The Convolutional Alternative
A **Convolutional Layer** dramatically solves this by imposing two core spatial biases:
1. **Local Connectivity (Sparse Interactions):** Instead of connecting a neuron to every pixel in the entire image, a convolutional neuron connects only to a localized spatial region of the input, called a **receptive field** or **filter kernel** of spatial size $K \times K$.
2. **Weight Sharing (Parameter Tiedness):** The exact same weights (filters) slide across all spatial positions in the image. Rather than having a different set of weights for every pixel coordinate, we reuse the same filter to detect a specific visual feature (like a diagonal edge) anywhere in the image.

The number of weights for a convolutional layer with $N$ filters (which yields $N$ channels in the output activation map) of spatial size $K \times K$ is:
$$\text{Params}_{\text{Conv}} = N \times (K \times K \times C_{in} + 1)$$

* **Using the HD Input Example with $N=1000$ filters of spatial size $K=5$ and $C_{in}=3$:**
  $$\text{Params}_{\text{Conv}} = 1000 \times (5 \times 5 \times 3 + 1) = 1000 \times (75 + 1) = \mathbf{76,000 \text{ parameters}}$$

By switching to a Convolutional Layer, we reduce the parameters from **3 billion** to **76 thousand** (a **$39,400\times$ reduction**), while keeping the ability to process HD images!

### 3. Visual Representation: MLP vs. CNN Spatial Biases
Below is the structural contrast between the MLP flattening paradigm and the CNN spatial grid preservation:

![MLP vs CNN Spatial Preservation](assets/lecture5_diagram_1.png)
